# Supervised Baselines

This notebook creates simple dummy-model baselines for the monetary relief classification task.
It is meant to contextualize the stronger supervised models by showing how much performance improves over trivial prediction rules.

## Setup

The baseline setup mirrors the main final clean development notebook as closely as possible:
- same data source
- same target definition
- same maximum modeling row cap
- same train/test split and random seed

The only models used here are `DummyClassifier(strategy="most_frequent")` and `DummyClassifier(strategy="stratified")`.

In [2]:
import os
from pathlib import Path
import warnings

import pandas as pd

from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "final_unsupervised_features_relief_vader.parquet"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "output_tables"
OUTPUT_TABLES_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR = PROJECT_ROOT / "data" / "processed"

POSITIVE_RELIEF_VALUES = ["Closed with monetary relief"]
RANDOM_STATE = 42
CV_FOLDS = 5
MAX_MODEL_ROWS = 120000
N_JOBS = min(4, max(1, (os.cpu_count() or 2) - 1))

assert DATA_PATH.exists(), f"Expected modeling parquet at {DATA_PATH}"

raw_df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(raw_df):,} rows and {raw_df.shape[1]} columns from {DATA_PATH.name}")
raw_df.head()

Loaded 397,575 rows and 44 columns from final_unsupervised_features_relief_vader.parquet


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,cleaned_consumer_narrative,processed_narrative,cluster,distance_to_centroid,dominant_topic,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,topic_8_prob,topic_9_prob,topic_10_prob,topic_11_prob,topic_12_prob,topic_13_prob,topic_14_prob,topic_15_prob,topic_16_prob,topic_17_prob,topic_18_prob,topic_19_prob,narrative_sentiment_score
0,2019-11-18,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,XXXX claimed they delivered a package to my ad...,NaN,DISCOVER BANK,MA,021XX,NaN,Consent provided,Web,2019-11-18,Closed with explanation,True,N/A,3442136,REDACTED claimed they delivered a package to m...,claimed delivered package address never receiv...,11,0.992864,10,0.000602,0.000602,0.000602,0.000602,0.000602,0.000602,0.000602,0.000602,0.000602,0.154693,0.712191,0.000602,0.061553,0.000602,0.000602,0.000602,0.000602,0.000602,0.061924,0.000602,0.4019
1,2020-04-10,Credit card or prepaid card,General-purpose prepaid card,Trouble using the card,Trouble getting information about the card,I got a Brinks Money pre-paid card in the mail...,Company has responded to the consumer and the ...,Netspend Corporation,IL,60657,NaN,Consent provided,Web,2020-04-14,Closed with explanation,True,N/A,3601853,I got a Brinks Money pre-paid card in the mail...,got brink money pre paid card mail assuming un...,3,0.998468,8,0.002174,0.002174,0.276566,0.002174,0.002174,0.002174,0.002174,0.002174,0.503805,0.002174,0.002174,0.002174,0.002174,0.182673,0.002174,0.002174,0.002174,0.002174,0.002174,0.002174,0.0000
2,2019-07-09,Credit card or prepaid card,Store credit card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,On XX/XX/XXXX I was called by a creditor Nelso...,NaN,Nelson Cruz & Associates LLC,TN,37043,NaN,Consent provided,Web,2019-07-09,Closed with explanation,True,N/A,3300820,On REDACTED_DATE I was called by a creditor Ne...,called creditor nelson cruz associate claimed ...,44,0.978543,9,0.000926,0.085328,0.147042,0.000926,0.000926,0.000926,0.000926,0.068550,0.186560,0.386073,0.000926,0.077052,0.000926,0.000926,0.000926,0.000926,0.037359,0.000926,0.000926,0.000926,-0.8002
3,2020-07-10,Credit card or prepaid card,General-purpose credit card or charge card,Trouble using your card,Can't use card to make purchases,Around XX/XX/2020 i XXXX XXXX XXXX opened a cr...,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,937XX,NaN,Consent provided,Web,2020-07-10,Closed with explanation,True,N/A,3739698,Around REDACTED / REDACTED /2020 i REDACTED RE...,around opened credit card account online capit...,25,0.852135,2,0.000538,0.000538,0.426388,0.000538,0.000538,0.000538,0.000538,0.000538,0.183611,0.320316,0.018011,0.000538,0.000538,0.000538,0.000538,0.000538,0.000538,0.000538,0.000538,0.043609,0.8060
4,2019-06-24,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,Equifax sent me credit card suggestions to hel...,NaN,"EQUIFAX, INC.",OR,971XX,NaN,Consent provided,Web,2019-06-24,Closed with explanation,True,N/A,3285243,Equifax sent me credit card suggestions to hel...,equifax sent credit card suggestion help impro...,44,0.963603,6,0.002083,0.002083,0.002083,0.002083,0.002083,0.179421,0.783079,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.002083,0.8591


## Prepare Target

The dummy baselines do not need the engineered predictors, but they should use the same target construction and same eligible modeling rows so the comparison is fair.

In [3]:
def score_estimator(estimator, X_test, y_test):
    y_score = estimator.predict_proba(X_test)[:, 1]
    y_pred = estimator.predict(X_test)
    return {
        "holdout_accuracy": accuracy_score(y_test, y_pred),
        "holdout_average_precision": average_precision_score(y_test, y_score),
        "holdout_roc_auc": roc_auc_score(y_test, y_score),
        "holdout_recall": recall_score(y_test, y_pred, zero_division=0),
        "holdout_precision": precision_score(y_test, y_pred, zero_division=0),
        "holdout_f1": f1_score(y_test, y_pred, zero_division=0),
        "holdout_balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "y_score": y_score,
        "y_pred": y_pred,
    }


model_df = raw_df.copy()
model_df["Date received"] = pd.to_datetime(model_df["Date received"], errors="coerce")
model_df["target_relief"] = model_df["Company response to consumer"].isin(POSITIVE_RELIEF_VALUES).astype(int)
model_df = model_df.dropna(subset=["Date received", "target_relief"]).copy()

if len(model_df) > MAX_MODEL_ROWS:
    _, model_df = train_test_split(
        model_df,
        test_size=MAX_MODEL_ROWS,
        stratify=model_df["target_relief"],
        random_state=RANDOM_STATE,
    )

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    stratify=model_df["target_relief"],
    random_state=RANDOM_STATE,
)

X_train = pd.DataFrame({"baseline_constant": 1}, index=train_df.index)
X_test = pd.DataFrame({"baseline_constant": 1}, index=test_df.index)
y_train = train_df["target_relief"]
y_test = test_df["target_relief"]

scoring = {
    "accuracy": "accuracy",
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy",
}
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"Modeling rows: {len(model_df):,}")
print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Overall relief rate: {model_df['target_relief'].mean() * 100:.3f}%")
print(f"Train relief rate: {y_train.mean() * 100:.3f}%")
print(f"Test relief rate: {y_test.mean() * 100:.3f}%")

Modeling rows: 120,000
Train rows: 96,000
Test rows: 24,000
Overall relief rate: 15.719%
Train relief rate: 15.719%
Test relief rate: 15.721%


## Dummy Baselines

`most_frequent` always predicts the majority class.
`stratified` predicts classes at random according to the observed class frequencies in the training data.

In [4]:
baseline_rows = []
holdout_frames = []

for model_name, strategy in [
    ("Dummy Most Frequent", "most_frequent"),
    ("Dummy Stratified", "stratified"),
]:
    estimator = DummyClassifier(strategy=strategy, random_state=RANDOM_STATE)
    cv_results = cross_validate(
        estimator,
        X_train,
        y_train,
        scoring=scoring,
        cv=cv,
        n_jobs=N_JOBS,
        return_train_score=False,
    )
    estimator.fit(X_train, y_train)
    holdout_scores = score_estimator(estimator, X_test, y_test)

    row = {
        "model_family": model_name,
        "strategy": strategy,
    }
    for metric in scoring:
        row[f"cv_mean_{metric}"] = cv_results[f"test_{metric}"].mean()
        row[f"cv_std_{metric}"] = cv_results[f"test_{metric}"].std()
    row.update({k: v for k, v in holdout_scores.items() if not k.startswith("y_")})
    baseline_rows.append(row)

    holdout_frame = test_df[["Complaint ID", "Product", "Issue", "Company", "target_relief"]].copy()
    holdout_frame["model_family"] = model_name
    holdout_frame["predicted_label"] = holdout_scores["y_pred"]
    holdout_frame["score"] = holdout_scores["y_score"]
    holdout_frames.append(holdout_frame)

baseline_results_df = pd.DataFrame(baseline_rows).sort_values(by="holdout_f1", ascending=False).reset_index(drop=True)
baseline_holdout_df = pd.concat(holdout_frames, axis=0).reset_index(drop=True)

comparison_columns = [
    "model_family",
    "strategy",
    "cv_mean_accuracy",
    "cv_std_accuracy",
    "cv_mean_average_precision",
    "cv_std_average_precision",
    "cv_mean_roc_auc",
    "cv_std_roc_auc",
    "cv_mean_recall",
    "cv_std_recall",
    "cv_mean_precision",
    "cv_std_precision",
    "cv_mean_f1",
    "cv_std_f1",
    "holdout_accuracy",
    "holdout_average_precision",
    "holdout_roc_auc",
    "holdout_recall",
    "holdout_precision",
    "holdout_f1",
    "holdout_balanced_accuracy",
]

display(baseline_results_df[comparison_columns].round(4))

baseline_results_path = OUTPUT_TABLES_DIR / "03a_supervised_baseline_results.csv"
baseline_results_df.to_csv(baseline_results_path, index=False)

baseline_holdout_path = ARTIFACT_DIR / "03a_supervised_baseline_holdout_predictions.parquet"
baseline_holdout_df.to_parquet(baseline_holdout_path, index=False)

print(f"Saved baseline comparison table to {baseline_results_path}")
print(f"Saved baseline holdout predictions to {baseline_holdout_path}")

,model_family,strategy,cv_mean_accuracy,cv_std_accuracy,cv_mean_average_precision,cv_std_average_precision,cv_mean_roc_auc,cv_std_roc_auc,cv_mean_recall,cv_std_recall,cv_mean_precision,cv_std_precision,cv_mean_f1,cv_std_f1,holdout_accuracy,holdout_average_precision,holdout_roc_auc,holdout_recall,holdout_precision,holdout_f1,holdout_balanced_accuracy
0,Dummy Stratified,stratified,0.7360,0.0017,0.1575,0.0009,0.5009,0.0032,0.158,0.0055,0.1587,0.0055,0.1584,0.0055,0.7309,0.1558,0.4943,0.1492,0.1477,0.1485,0.4943
1,Dummy Most Frequent,most_frequent,0.8428,0.0000,0.1572,0.0000,0.5000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.8428,0.1572,0.5000,0.0000,0.0000,0.0000,0.5000


Saved baseline comparison table to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\output_tables\03a_supervised_baseline_results.csv
Saved baseline holdout predictions to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\03a_supervised_baseline_holdout_predictions.parquet
